# AURORA ? MAX V4.1 ? Hardened Meta Champion

V4 passed the first real champion test. V4.1 keeps the same validation-safe philosophy, but hardens the parts that still matter before trusting a production flag:

- fund / ETF / trust canaries are enforced directly (`ABALX`, fund share classes, commodity/crypto products);
- valid operating tickers ending in `X` stay in the universe (`CVX`, `BSX`, `BDX`, `EQIX`, `X` when present);
- feature leakage canaries fail fast if a forward, target, price target, lens prediction, or pseudo-future field enters the residual model;
- champion selection still happens only on 2019-2020 tune score, never on 2021+ validation;
- validation gates now require per-year MAE and IC stability, not only pooled validation improvement;
- confidence means lower model-disagreement and must show lower error plus positive rank signal.

Promotion in V4.1 means: a tune-selected champion beats the deterministic spine, uniform blend, and best single lens on mature validation, survives filter/leakage canaries, and wins in each primary validation year.


## 1. Runtime, Repo, Config


In [ ]:
import os, sys, json, time, random, subprocess, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
    import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, RandomForestRegressor

warnings.filterwarnings("ignore", category=UserWarning)

REPO_URL = "https://github.com/tbasaure-sys/fin.git"
REPO_REF = "main"
WORKDIR = Path("/content/fin") if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path("/content/drive/MyDrive/blsprime_aurora_omega") if IN_COLAB else Path("./_local_data/blsprime_aurora_omega")
PANEL_ROOT = DRIVE_ROOT / "panel"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"

START_YEAR = 2005
LAST_FEATURE_YEAR = 2024
DATA_CUTOFF_DATE = pd.Timestamp.utcnow().tz_localize(None).date().isoformat()
CORE_END_YEAR = 2018
TUNE_START_YEAR = 2019
TUNE_END_YEAR = 2020
VAL_START_YEAR = 2021
HORIZON_YEARS = 3
TARGET = "ann_return_3y_fwd"
SEED = 7
NOTEBOOK_VERSION = "aurora_omega_max_v4_1_hardened_meta_champion"
ARTIFACT_NAME_PREFIX = "omega_v4_1_hardened_meta_champion"
MIN_PRIMARY_YEAR_ROWS = 100


for p in [DRIVE_ROOT, PANEL_ROOT, ARTIFACT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("Runtime:", {"python": sys.version.split()[0], "cuda": torch.cuda.is_available(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
print("Drive:", DRIVE_ROOT)
print("Data cutoff:", DATA_CUTOFF_DATE)


## 2. Sync Repo and Imports


In [ ]:
if IN_COLAB:
    if not WORKDIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(WORKDIR)])
    subprocess.check_call(["git", "-C", str(WORKDIR), "fetch", "origin", REPO_REF])
    subprocess.check_call(["git", "-C", str(WORKDIR), "checkout", REPO_REF])
    subprocess.check_call(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", REPO_REF])

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

import importlib
import scripts.run_aurora_router_local as router
importlib.reload(router)
from aurora_omega.data import LENS_NAMES

print("Repo:", WORKDIR)
print("Lenses:", LENS_NAMES)


## 3. Rebuild Featured Panel and Harden Universe Filter


In [ ]:
PANEL_CANDIDATES = [
    PANEL_ROOT / "panel_autodiscover_2005_2024_1500.parquet",
    PANEL_ROOT / "panel_cache_only_2005_2024_1500.parquet",
    PANEL_ROOT / "panel_selfcontained_2005_2024_1500.parquet",
]
panel_path = next((p for p in PANEL_CANDIDATES if p.exists() and p.stat().st_size > 0), None)
if panel_path is None:
    raise FileNotFoundError("No raw panel found. Expected one of: " + ", ".join(map(str, PANEL_CANDIDATES)))

panel = pd.read_parquet(panel_path)
print("Loaded raw panel:", panel_path, panel.shape, "tickers:", panel["ticker"].nunique())

featured = router.add_features(panel.copy())
featured = router.add_lens_predictions(featured)
featured["omega_regime"] = featured.apply(router.classify_spine_regime, axis=1)
featured["omega_primary_question"] = featured["omega_regime"].map(router.primary_question_for_regime)
expectations = featured.apply(router.reverse_dcf_expectations, axis=1)
featured["omega_expectations_pressure"] = [e.get("valuation_pressure_score", np.nan) for e in expectations]
featured["omega_feasibility_score"] = [
    router.score_expectation_feasibility(row, e).get("score", np.nan)
    for (_, row), e in zip(featured.iterrows(), expectations)
]
featured["omega_downside_anchor_score"] = [
    router.anchor_lens_checks(row, router.classify_spine_regime(row), e).get("asset_value", {}).get("score", np.nan)
    for (_, row), e in zip(featured.iterrows(), expectations)
]

MUST_EXCLUDE_PRODUCT_TICKERS = {
    "ABALX", "FNILX", "VTSAX", "VBTIX", "GBTC", "ETHE", "IBIT", "FBTC", "BITB", "ARKB",
    "SLV", "GLD", "IAU", "USO", "UNG", "SPY", "QQQ", "VOO", "VTI", "IWM", "DIA",
    "TLT", "HYG", "LQD", "BND", "SHY", "IEF", "EEM", "EFA", "XLF", "XLK", "XLE", "XLV",
}
VALID_OPERATING_CANARY_KEEP = {"CVX", "BSX", "BDX", "EQIX", "X"}


def common_operating_equity_mask(frame):
    ticker = frame["ticker"].astype(str).str.upper().str.strip()
    sector = frame.get("sector", pd.Series("Unknown", index=frame.index)).astype(str).str.lower()
    industry = frame.get("industry", pd.Series("Unknown", index=frame.index)).astype(str).str.lower()
    name = frame.get("company_name", pd.Series("", index=frame.index)).astype(str).str.lower()
    text = sector + " " + industry + " " + name

    # Keep ordinary US operating tickers, including class-share forms like BRK-B/BRK.B.
    operating_symbol = ticker.str.match(r"^[A-Z]{1,5}([.-][A-Z])?$")
    blank_sector = sector.isin(["", "unknown", "nan", "none"])

    fund_like_text = text.str.contains(
        r"mutual fund|index fund|exchange traded fund|\betf\b|closed-end|target date|money market|portfolio fund|"
        r"open-end|balanced fund|income fund|growth fund|bond fund|large cap fund|small cap fund|"
        r"ishares|vanguard fund|fidelity fund|blackrock fund|bitcoin trust|ethereum trust|grayscale",
        regex=True,
        na=False,
    )

    # US mutual fund share classes are commonly five letters ending in X. This catches ABALX even if FMP text is sparse.
    fund_family_share_class = ticker.str.len().eq(5) & ticker.str.endswith("X")
    known_product_ticker = ticker.isin(MUST_EXCLUDE_PRODUCT_TICKERS)
    return operating_symbol & ~blank_sector & ~fund_like_text & ~fund_family_share_class & ~known_product_ticker

pre_tickers = set(featured["ticker"].astype(str).str.upper())
mask = common_operating_equity_mask(featured)
removed_sample = sorted(set(featured.loc[~mask, "ticker"].astype(str).str.upper()))[:60]
pre_shape = featured.shape
featured = featured.loc[mask].sort_values(["ticker", "year"]).reset_index(drop=True)
post_tickers = set(featured["ticker"].astype(str).str.upper())

unexpected_product_survivors = sorted((MUST_EXCLUDE_PRODUCT_TICKERS & pre_tickers) & post_tickers)
wrongly_removed_operating_canaries = sorted((VALID_OPERATING_CANARY_KEEP & pre_tickers) - post_tickers)
filter_audit = {
    "pre_rows": int(pre_shape[0]),
    "post_rows": int(len(featured)),
    "pre_tickers": int(len(pre_tickers)),
    "post_tickers": int(featured["ticker"].nunique()),
    "removed_rows": int(pre_shape[0] - len(featured)),
    "removed_tickers_sample": removed_sample,
    "must_exclude_product_survivors": unexpected_product_survivors,
    "wrongly_removed_operating_canaries": wrongly_removed_operating_canaries,
    "fund_share_class_rule": "exclude exactly 5-letter tickers ending in X, while keeping normal operating X tickers",
}
print(json.dumps(filter_audit, indent=2))
if unexpected_product_survivors:
    raise AssertionError(f"Product/fund canaries survived common-equity filter: {unexpected_product_survivors}")
if wrongly_removed_operating_canaries:
    raise AssertionError(f"Operating canaries were removed by common-equity filter: {wrongly_removed_operating_canaries}")

featured_path = PANEL_ROOT / "featured_panel_v4_1_hardened_operating_equity.parquet"
featured.to_parquet(featured_path, index=False)
print("Saved:", featured_path)
display(featured[["ticker", "year", "sector", "industry", "omega_regime", "pred_reverseDcf", "pred_assetValue", TARGET]].head())


## 4. Mature Target and Splits


In [ ]:
def mask_immature_forward_returns(frame, target_col=TARGET, horizon_years=HORIZON_YEARS, cutoff_date=DATA_CUTOFF_DATE):
    out = frame.copy()
    asof = pd.to_datetime(out["asof_date"], errors="coerce")
    cutoff = pd.Timestamp(cutoff_date)
    if cutoff.tzinfo is not None:
        cutoff = cutoff.tz_localize(None)
    mature_date = asof + pd.DateOffset(years=horizon_years)
    matured = mature_date.notna() & (mature_date <= cutoff)
    out[f"{target_col}_matured"] = matured
    out.loc[~matured, target_col] = np.nan
    return out

data = mask_immature_forward_returns(featured)
for c in [TARGET, "year"]:
    data[c] = pd.to_numeric(data[c], errors="coerce")
data["year"] = data["year"].astype("Int64")

ACTIVE_3Y_LENSES = [name for name in LENS_NAMES if name != "capitalCycle" and f"pred_{name}" in data.columns]
lens_cols = [f"pred_{name}" for name in ACTIVE_3Y_LENSES]
for c in lens_cols:
    data[c] = pd.to_numeric(data[c], errors="coerce")

data = data.dropna(subset=["ticker", "year", TARGET] + lens_cols).copy()
data["year"] = data["year"].astype(int)

core_df = data[data["year"] <= CORE_END_YEAR].copy()
tune_df = data[data["year"].between(TUNE_START_YEAR, TUNE_END_YEAR)].copy()
val_df = data[data["year"] >= VAL_START_YEAR].copy()
val_year_counts = val_df.groupby("year").size().rename("rows").reset_index()
primary_val_years = val_year_counts.loc[val_year_counts["rows"] >= MIN_PRIMARY_YEAR_ROWS, "year"].astype(int).tolist()
primary_val_df = val_df[val_df["year"].isin(primary_val_years)].copy()

print("Rows:", {"core": len(core_df), "tune": len(tune_df), "all_mature_val": len(val_df), "primary_val": len(primary_val_df)})
print("Tickers:", {"core": core_df.ticker.nunique(), "tune": tune_df.ticker.nunique(), "primary_val": primary_val_df.ticker.nunique()})
print("Validation rows by year:")
display(val_year_counts)
print("Primary validation years:", primary_val_years)

if len(core_df) < 1000 or len(tune_df) < 300 or len(primary_val_df) < 300:
    raise RuntimeError("Not enough mature data for V4 gates.")


## 5. Metrics and Train-Only Spine


In [ ]:
def mae_np(pred, y):
    s = pd.DataFrame({"pred": pred, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    return float(np.mean(np.abs(s["pred"] - s["y"]))) if len(s) else float("nan")

def ic_np(pred, y):
    s = pd.DataFrame({"pred": pred, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 20 or s["pred"].nunique() < 5 or s["y"].nunique() < 5:
        return float("nan")
    return float(s["pred"].rank().corr(s["y"].rank()))

def decile_spread(frame, pred_col, target_col=TARGET):
    spreads = []
    for _, sub in frame[["year", pred_col, target_col]].dropna().groupby("year"):
        if len(sub) < 80 or sub[pred_col].nunique() < 10:
            continue
        q = pd.qcut(sub[pred_col], 10, labels=False, duplicates="drop")
        if q.max() < 1:
            continue
        spreads.append(float(sub.loc[q == q.max(), target_col].mean() - sub.loc[q == q.min(), target_col].mean()))
    return float(np.mean(spreads)) if spreads else float("nan")

def by_year_metrics(frame, pred_col, target_col=TARGET):
    rows = []
    for yr, sub in frame[["year", pred_col, target_col]].dropna().groupby("year"):
        if len(sub) < 20:
            continue
        rows.append({"year": int(yr), "rows": int(len(sub)), "mae": mae_np(sub[pred_col], sub[target_col]), "ic": ic_np(sub[pred_col], sub[target_col])})
    return pd.DataFrame(rows)

def prior_for_lenses(names):
    raw = []
    for n in names:
        if n == "reverseDcf": raw.append(0.36)
        elif n == "assetValue": raw.append(0.26)
        elif n == "residualIncome": raw.append(0.17)
        elif n == "roicFade": raw.append(0.08)
        elif n == "dcf": raw.append(0.07)
        elif n == "unitEconomics": raw.append(0.04)
        else: raw.append(0.02)
    raw = np.asarray(raw, dtype="float64")
    return raw / raw.sum()

def fit_simplex_spine(frame, lens_cols, target_col=TARGET, epochs=2500, lr=0.05, l2=0.04):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    X = torch.tensor(frame[lens_cols].values.astype("float32"), device=device)
    y = torch.tensor(frame[target_col].values.astype("float32"), device=device)
    prior = torch.tensor(prior_for_lenses([c.replace("pred_", "") for c in lens_cols]).astype("float32"), device=device)
    theta = torch.zeros(len(lens_cols), device=device, requires_grad=True)
    opt = torch.optim.Adam([theta], lr=lr)
    for _ in range(epochs):
        w = torch.softmax(theta, dim=0)
        pred = X @ w
        loss = torch.nn.functional.smooth_l1_loss(pred, y, beta=0.04) + l2 * ((w - prior) ** 2).sum()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    with torch.no_grad():
        w = torch.softmax(theta, dim=0).cpu().numpy()
    return {col.replace("pred_", ""): float(weight) for col, weight in zip(lens_cols, w)}

spine_weights = fit_simplex_spine(core_df, lens_cols)

def apply_spine(frame, weights):
    pred = np.zeros(len(frame), dtype="float64")
    for name, weight in weights.items():
        pred += weight * frame[f"pred_{name}"].astype(float).values
    return pred

for frame in [core_df, tune_df, val_df, primary_val_df]:
    frame["spine_pred"] = apply_spine(frame, spine_weights)
    frame["uniform_pred"] = frame[lens_cols].mean(axis=1).astype(float)
    lens_values = frame[lens_cols].astype(float)
    frame["lens_mean"] = lens_values.mean(axis=1)
    frame["lens_std"] = lens_values.std(axis=1)
    frame["lens_range"] = lens_values.max(axis=1) - lens_values.min(axis=1)
    frame["reverse_minus_spine"] = frame.get("pred_reverseDcf", frame["spine_pred"]) - frame["spine_pred"]
    frame["asset_minus_spine"] = frame.get("pred_assetValue", frame["spine_pred"]) - frame["spine_pred"]

print("Spine weights:", json.dumps(spine_weights, indent=2))
for label, frame in [("core", core_df), ("tune", tune_df), ("primary_val", primary_val_df), ("all_mature_val", val_df)]:
    single = {c.replace("pred_", ""): mae_np(frame[c], frame[TARGET]) for c in lens_cols}
    print(label, {
        "spine_mae": mae_np(frame["spine_pred"], frame[TARGET]),
        "uniform_mae": mae_np(frame["uniform_pred"], frame[TARGET]),
        "best_single": min(single.items(), key=lambda kv: kv[1]),
        "spine_ic": ic_np(frame["spine_pred"], frame[TARGET]),
        "spine_decile": decile_spread(frame, "spine_pred"),
    })


## 6. Feature Matrix and Leakage Canary


In [ ]:
def make_model_frame(frame):
    forbidden = {TARGET, "year"}
    forbidden_prefixes = ("ann_return_", "price_t", "target_", "future_", "hard_", "omega_weight_", "pred_")
    numeric = []
    for c in frame.columns:
        if c in forbidden:
            continue
        if any(str(c).startswith(p) for p in forbidden_prefixes):
            continue
        if pd.api.types.is_numeric_dtype(frame[c]) and frame[c].notna().sum() >= 50:
            numeric.append(c)
    cat = [c for c in ["omega_regime", "sector", "industry"] if c in frame.columns]
    return numeric, cat

feature_cols, cat_cols = make_model_frame(core_df)
leakage_prefixes = ("ann_return_", "price_t", "target_", "future_", "hard_", "omega_weight_", "pred_")
leakage_features = [c for c in feature_cols if c == TARGET or any(str(c).startswith(p) for p in leakage_prefixes)]
feature_audit = {
    "feature_count": int(len(feature_cols)),
    "categorical_features": cat_cols,
    "leakage_feature_count": int(len(leakage_features)),
    "leakage_features": leakage_features[:30],
}
print(json.dumps(feature_audit, indent=2))
print("Feature cols sample:", feature_cols[:35])
if leakage_features:
    raise AssertionError(f"Forward/leaky fields entered model features: {leakage_features}")

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), feature_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cat_cols),
    ],
    remainder="drop",
)

X_core_raw = core_df[feature_cols + cat_cols]
X_tune_raw = tune_df[feature_cols + cat_cols]
X_val_raw = val_df[feature_cols + cat_cols]
X_primary_raw = primary_val_df[feature_cols + cat_cols]

y_core = core_df[TARGET].astype(float).values
y_tune = tune_df[TARGET].astype(float).values
y_val = val_df[TARGET].astype(float).values
y_primary = primary_val_df[TARGET].astype(float).values
resid_core = y_core - core_df["spine_pred"].astype(float).values


## 7. Residual Challenger Library


In [ ]:
challengers = {
    "hgb_abs_shallow": HistGradientBoostingRegressor(loss="absolute_error", learning_rate=0.035, max_iter=260, max_leaf_nodes=18, l2_regularization=0.08, random_state=SEED),
    "hgb_abs_deep": HistGradientBoostingRegressor(loss="absolute_error", learning_rate=0.025, max_iter=360, max_leaf_nodes=28, l2_regularization=0.12, random_state=SEED + 1),
    "hgb_sq": HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.030, max_iter=260, max_leaf_nodes=18, l2_regularization=0.18, random_state=SEED + 2),
    "rf_stable": RandomForestRegressor(n_estimators=260, max_depth=8, min_samples_leaf=18, random_state=SEED, n_jobs=-1),
    "rf_smoother": RandomForestRegressor(n_estimators=360, max_depth=6, min_samples_leaf=28, random_state=SEED + 3, n_jobs=-1),
    "extra_trees": ExtraTreesRegressor(n_estimators=320, max_depth=7, min_samples_leaf=18, random_state=SEED, n_jobs=-1),
    "ridge": Ridge(alpha=8.0, random_state=SEED),
    "elastic": ElasticNet(alpha=0.002, l1_ratio=0.10, random_state=SEED, max_iter=10000),
    "huber": HuberRegressor(alpha=0.004, epsilon=1.35, max_iter=1200),
}

def tune_metrics(pred, y, frame_for_decile=None, pred_col_name="_tmp_pred"):
    out = {"mae": mae_np(pred, y), "ic": ic_np(pred, y)}
    if frame_for_decile is not None:
        tmp = frame_for_decile.copy()
        tmp[pred_col_name] = pred
        out["decile"] = decile_spread(tmp, pred_col_name)
    else:
        out["decile"] = float("nan")
    out["score"] = out["mae"] - 0.030 * (0 if not np.isfinite(out["ic"]) else out["ic"]) - 0.020 * (0 if not np.isfinite(out["decile"]) else out["decile"])
    return out

def tune_blend(anchor, residual_pred, y, frame):
    best = None
    for rho in np.linspace(0.0, 0.80, 33):
        pred = anchor + rho * residual_pred
        row = {"rho": float(rho), **tune_metrics(pred, y, frame)}
        if best is None or row["score"] < best["score"]:
            best = row
    return best

pred_bank = {}
rows = []
for name, model in challengers.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    pipe.fit(X_core_raw, resid_core)
    tune_resid = pipe.predict(X_tune_raw)
    val_resid = pipe.predict(X_val_raw)
    primary_resid = pipe.predict(X_primary_raw)
    blend = tune_blend(tune_df["spine_pred"].astype(float).values, tune_resid, y_tune, tune_df)
    preds = {
        "tune": tune_df["spine_pred"].astype(float).values + blend["rho"] * tune_resid,
        "val": val_df["spine_pred"].astype(float).values + blend["rho"] * val_resid,
        "primary": primary_val_df["spine_pred"].astype(float).values + blend["rho"] * primary_resid,
    }
    pred_bank[name] = preds
    primary_val_df[f"pred_{name}"] = preds["primary"]
    val_df[f"pred_{name}"] = preds["val"]
    tune_df[f"pred_{name}"] = preds["tune"]
    row = {
        "model": name,
        "rho": blend["rho"],
        "tune_mae": tune_metrics(preds["tune"], y_tune, tune_df)["mae"],
        "tune_ic": tune_metrics(preds["tune"], y_tune, tune_df)["ic"],
        "tune_decile": tune_metrics(preds["tune"], y_tune, tune_df)["decile"],
        "tune_score": blend["score"],
        "val_mae": mae_np(preds["primary"], y_primary),
        "val_ic": ic_np(preds["primary"], y_primary),
        "val_decile": decile_spread(primary_val_df.assign(_p=preds["primary"]), "_p"),
        "all_mature_val_mae": mae_np(preds["val"], y_val),
    }
    rows.append(row)

single_results = pd.DataFrame(rows).sort_values("tune_score")
display(single_results)


## 8. Tune-Selected Meta-Ensembles


In [ ]:
top_models = list(single_results.head(5)["model"])
ensemble_rows = []
ensemble_bank = {}

def eval_candidate(name, tune_pred, primary_pred, val_pred):
    tune_m = tune_metrics(tune_pred, y_tune, tune_df)
    row = {
        "model": name,
        "rho": None,
        "tune_mae": tune_m["mae"],
        "tune_ic": tune_m["ic"],
        "tune_decile": tune_m["decile"],
        "tune_score": tune_m["score"],
        "val_mae": mae_np(primary_pred, y_primary),
        "val_ic": ic_np(primary_pred, y_primary),
        "val_decile": decile_spread(primary_val_df.assign(_p=primary_pred), "_p"),
        "all_mature_val_mae": mae_np(val_pred, y_val),
    }
    ensemble_bank[name] = {"tune": tune_pred, "primary": primary_pred, "val": val_pred}
    ensemble_rows.append(row)

# include singles in the meta selection pool
for name in single_results["model"]:
    eval_candidate(name, pred_bank[name]["tune"], pred_bank[name]["primary"], pred_bank[name]["val"])

# pairwise convex ensembles selected by tune score
for i, a in enumerate(top_models):
    for b in top_models[i + 1:]:
        best = None
        for wa in np.linspace(0.0, 1.0, 21):
            wb = 1.0 - wa
            tune_pred = wa * pred_bank[a]["tune"] + wb * pred_bank[b]["tune"]
            m = tune_metrics(tune_pred, y_tune, tune_df)
            if best is None or m["score"] < best["score"]:
                best = {"wa": float(wa), "wb": float(wb), **m}
        name = f"ens_{a}_{b}_{best['wa']:.2f}_{best['wb']:.2f}"
        primary_pred = best["wa"] * pred_bank[a]["primary"] + best["wb"] * pred_bank[b]["primary"]
        val_pred = best["wa"] * pred_bank[a]["val"] + best["wb"] * pred_bank[b]["val"]
        tune_pred = best["wa"] * pred_bank[a]["tune"] + best["wb"] * pred_bank[b]["tune"]
        eval_candidate(name, tune_pred, primary_pred, val_pred)

# equal-weight top-k ensemble variants
for k in [2, 3, 4, 5]:
    names = top_models[:k]
    name = "ens_equal_top" + str(k)
    tune_pred = np.mean([pred_bank[n]["tune"] for n in names], axis=0)
    primary_pred = np.mean([pred_bank[n]["primary"] for n in names], axis=0)
    val_pred = np.mean([pred_bank[n]["val"] for n in names], axis=0)
    eval_candidate(name, tune_pred, primary_pred, val_pred)

all_results = pd.DataFrame(ensemble_rows).sort_values("tune_score")
display(all_results.head(20))
champion_name = str(all_results.iloc[0]["model"])
champion_pred_primary = ensemble_bank[champion_name]["primary"]
champion_pred_val = ensemble_bank[champion_name]["val"]
champion_pred_tune = ensemble_bank[champion_name]["tune"]
primary_val_df["champion_pred"] = champion_pred_primary
val_df["champion_pred"] = champion_pred_val
tune_df["champion_pred"] = champion_pred_tune
print("Tune-selected champion:", champion_name)


## 9. Confidence and Abstention Diagnostics


In [ ]:
# Agreement across top models is a useful uncertainty proxy. Lower dispersion should be safer.
agreement_models = top_models[:5]
tune_stack = np.vstack([pred_bank[n]["tune"] for n in agreement_models]).T
primary_stack = np.vstack([pred_bank[n]["primary"] for n in agreement_models]).T
val_stack = np.vstack([pred_bank[n]["val"] for n in agreement_models]).T

tune_agreement = tune_stack.std(axis=1)
primary_agreement = primary_stack.std(axis=1)
val_agreement = val_stack.std(axis=1)
tune_df["agreement_std"] = tune_agreement
primary_val_df["agreement_std"] = primary_agreement
val_df["agreement_std"] = val_agreement

agreement_cut = float(np.quantile(tune_agreement, 0.60))
primary_val_df["high_confidence"] = primary_val_df["agreement_std"] <= agreement_cut
val_df["high_confidence"] = val_df["agreement_std"] <= agreement_cut

hc = primary_val_df["high_confidence"]
confidence_report = {
    "agreement_models": agreement_models,
    "agreement_cut_tune_p60": agreement_cut,
    "primary_high_conf_rows": int(hc.sum()),
    "primary_high_conf_share": float(hc.mean()) if len(hc) else 0.0,
    "primary_high_conf_mae": mae_np(primary_val_df.loc[hc, "champion_pred"], primary_val_df.loc[hc, TARGET]) if hc.any() else None,
    "primary_all_mae": mae_np(primary_val_df["champion_pred"], primary_val_df[TARGET]),
    "primary_high_conf_ic": ic_np(primary_val_df.loc[hc, "champion_pred"], primary_val_df.loc[hc, TARGET]) if hc.any() else None,
    "primary_all_ic": ic_np(primary_val_df["champion_pred"], primary_val_df[TARGET]),
    "primary_high_conf_decile_spread": decile_spread(primary_val_df.loc[hc].copy(), "champion_pred") if hc.any() else None,
    "primary_all_decile_spread": decile_spread(primary_val_df, "champion_pred"),
}
print(json.dumps(confidence_report, indent=2))


## 10. Scorecard and Gates


In [ ]:
lens_mae_val = {name: mae_np(primary_val_df[f"pred_{name}"], primary_val_df[TARGET]) for name in ACTIVE_3Y_LENSES}
best_single_name, best_single_mae = min(lens_mae_val.items(), key=lambda kv: kv[1])

scorecard = {
    "version": NOTEBOOK_VERSION,
    "champion": champion_name,
    "selection_rule": "minimum tune_score only; validation not used for model selection",
    "val_rows": int(len(primary_val_df)),
    "val_years": sorted([int(x) for x in primary_val_df["year"].unique()]),
    "all_mature_val_rows": int(len(val_df)),
    "all_mature_val_years": sorted([int(x) for x in val_df["year"].unique()]),
    "champion_mae": mae_np(primary_val_df["champion_pred"], primary_val_df[TARGET]),
    "spine_mae": mae_np(primary_val_df["spine_pred"], primary_val_df[TARGET]),
    "uniform_mae": mae_np(primary_val_df["uniform_pred"], primary_val_df[TARGET]),
    "best_single_lens": best_single_name,
    "best_single_mae": best_single_mae,
    "champion_ic": ic_np(primary_val_df["champion_pred"], primary_val_df[TARGET]),
    "spine_ic": ic_np(primary_val_df["spine_pred"], primary_val_df[TARGET]),
    "uniform_ic": ic_np(primary_val_df["uniform_pred"], primary_val_df[TARGET]),
    "champion_decile_spread": decile_spread(primary_val_df, "champion_pred"),
    "spine_decile_spread": decile_spread(primary_val_df, "spine_pred"),
    "all_mature_champion_mae": mae_np(val_df["champion_pred"], val_df[TARGET]),
    "all_mature_champion_ic": ic_np(val_df["champion_pred"], val_df[TARGET]),
    "confidence": confidence_report,
    "lens_mae_val": lens_mae_val,
    "spine_weights": spine_weights,
    "filter_audit": filter_audit,
    "feature_audit": feature_audit,
}

by_year = {}
for col in ["champion_pred", "spine_pred", "uniform_pred"]:
    by_year[col] = by_year_metrics(val_df, col).to_dict(orient="records")

primary_year_scorecard = []
for yr in scorecard["val_years"]:
    sub = primary_val_df[primary_val_df["year"] == yr].copy()
    primary_year_scorecard.append({
        "year": int(yr),
        "rows": int(len(sub)),
        "champion_mae": mae_np(sub["champion_pred"], sub[TARGET]),
        "spine_mae": mae_np(sub["spine_pred"], sub[TARGET]),
        "uniform_mae": mae_np(sub["uniform_pred"], sub[TARGET]),
        "champion_ic": ic_np(sub["champion_pred"], sub[TARGET]),
        "spine_ic": ic_np(sub["spine_pred"], sub[TARGET]),
        "uniform_ic": ic_np(sub["uniform_pred"], sub[TARGET]),
    })
scorecard["primary_year_scorecard"] = primary_year_scorecard


def beats(a, b, margin):
    return np.isfinite(a) and np.isfinite(b) and a < b - margin

beats_baselines_each_year = all(
    row["champion_mae"] < min(row["spine_mae"], row["uniform_mae"]) - 0.0005
    for row in primary_year_scorecard
)
positive_ic_each_year = all(
    np.isfinite(row["champion_ic"]) and row["champion_ic"] > 0.010
    for row in primary_year_scorecard
)

hc_mae = confidence_report["primary_high_conf_mae"]
hc_ic = confidence_report["primary_high_conf_ic"]
gates = {
    "selection_does_not_peek_validation": True,
    "filter_product_canaries_clean": len(filter_audit["must_exclude_product_survivors"]) == 0,
    "filter_operating_canaries_kept": len(filter_audit["wrongly_removed_operating_canaries"]) == 0,
    "feature_leakage_canary_clean": feature_audit["leakage_feature_count"] == 0,
    "mature_validation_rows": scorecard["val_rows"] >= 800,
    "at_least_two_primary_validation_years": len(scorecard["val_years"]) >= 2,
    "beats_spine_mae": beats(scorecard["champion_mae"], scorecard["spine_mae"], 0.0020),
    "beats_uniform_mae": beats(scorecard["champion_mae"], scorecard["uniform_mae"], 0.0020),
    "beats_best_single_mae": beats(scorecard["champion_mae"], scorecard["best_single_mae"], 0.0015),
    "beats_baselines_each_primary_year": beats_baselines_each_year,
    "positive_champion_ic": np.isfinite(scorecard["champion_ic"]) and scorecard["champion_ic"] > 0.025,
    "positive_ic_each_primary_year": positive_ic_each_year,
    "ic_lift_vs_spine": np.isfinite(scorecard["champion_ic"]) and np.isfinite(scorecard["spine_ic"]) and scorecard["champion_ic"] > scorecard["spine_ic"] + 0.020,
    "positive_decile_spread": np.isfinite(scorecard["champion_decile_spread"]) and scorecard["champion_decile_spread"] > 0.030,
    "high_confidence_error_lift": (
        confidence_report["primary_high_conf_rows"] >= 150
        and hc_mae is not None
        and hc_mae <= scorecard["champion_mae"] - 0.005
    ),
    "high_confidence_positive_ic": hc_ic is not None and np.isfinite(hc_ic) and hc_ic > 0.015,
}
production_candidate = bool(all(gates.values()))
product_mode = "hardened_meta_residual_champion" if production_candidate else "spine_reverse_dcf_memo_with_ml_shadow"

print(json.dumps({"scorecard": scorecard, "gates": gates, "production_candidate": production_candidate}, indent=2)[:18000])
print("By-year:", json.dumps(by_year, indent=2)[:9000])


## 11. Export Artifact and Memos


In [ ]:
out_dir = ARTIFACT_ROOT / f"{ARTIFACT_NAME_PREFIX}_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')}"
out_dir.mkdir(parents=True, exist_ok=True)

validation_export = val_df[["ticker", "year", "asof_date", TARGET, "champion_pred", "spine_pred", "uniform_pred", "agreement_std", "high_confidence"] + lens_cols].copy()
validation_export.to_csv(out_dir / "validation_predictions.csv", index=False)
all_results.to_csv(out_dir / "tournament_results.csv", index=False)
(out_dir / "by_year_metrics.json").write_text(json.dumps(by_year, indent=2), encoding="utf-8")
(out_dir / "filter_audit.json").write_text(json.dumps(filter_audit, indent=2), encoding="utf-8")
(out_dir / "feature_audit.json").write_text(json.dumps(feature_audit, indent=2), encoding="utf-8")

manifest = {
    "version": NOTEBOOK_VERSION,
    "created_at": pd.Timestamp.utcnow().isoformat(),
    "artifact_dir": str(out_dir),
    "data_cutoff_date": DATA_CUTOFF_DATE,
    "production_candidate": production_candidate,
    "product_mode": product_mode,
    "gates": gates,
    "scorecard": scorecard,
    "filter_audit": filter_audit,
    "feature_audit": feature_audit,
    "single_challengers": single_results.to_dict(orient="records"),
    "all_challengers": all_results.head(30).to_dict(orient="records"),
    "decision": "PROMOTE_HARDENED_META_CHAMPION" if production_candidate else "DO_NOT_PROMOTE_ML_USE_SPINE_MEMO",
}
(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")


def primary_question(regime):
    return {
        "expensive_compounder": "Are market-implied expectations feasible?",
        "quality_compounder": "How long can excess ROIC persist?",
        "asset_heavy_cyclical": "Where are we in the supply response cycle?",
        "financial_book_capital": "Does book capital create value above cost of equity?",
        "commodity_resource": "Is normalized commodity economics better than spot expectations?",
        "pre_profit_platform": "Are unit economics improving enough to justify optionality?",
        "bottleneck_oligopoly": "Is scarcity durable and monetizable?",
        "regulated_utility_infrastructure": "Is regulated spread adequate versus cost of capital?",
    }.get(str(regime), "Which valuation question deserves trust first?")

memos = []
latest = val_df.sort_values(["ticker", "year"]).groupby("ticker", as_index=False).tail(1)
for _, row in latest.head(700).iterrows():
    regime = row.get("omega_regime", "general_intrinsic")
    selected = float(row["champion_pred"]) if production_candidate else float(row["spine_pred"])
    memo = {
        "ticker": str(row["ticker"]),
        "year": int(row["year"]),
        "asof_date": str(row.get("asof_date", "")),
        "version": NOTEBOOK_VERSION,
        "production_candidate": production_candidate,
        "product_mode": product_mode,
        "regime": str(regime),
        "primary_question": primary_question(regime),
        "selected_prediction_3y": selected,
        "champion_prediction_3y": float(row["champion_pred"]),
        "spine_prediction_3y": float(row["spine_pred"]),
        "reverse_dcf_prediction_3y": float(row.get("pred_reverseDcf", np.nan)),
        "asset_value_prediction_3y": float(row.get("pred_assetValue", np.nan)),
        "agreement_std": float(row.get("agreement_std", np.nan)),
        "high_confidence": bool(row.get("high_confidence", False)),
        "falsifiers": [
            "Market-implied growth does not show up in reported revenue or backlog.",
            "Margins/ROIC fade faster than the dominant lens assumes.",
            "Capital cycle supply response destroys pricing power.",
            "Balance-sheet or asset-value support is weaker than the downside case assumes.",
        ],
    }
    memos.append(memo)

mri_path = out_dir / "valuation_memos_v4_1.jsonl"
with open(mri_path, "w", encoding="utf-8") as f:
    for memo in memos:
        f.write(json.dumps(memo) + "\n")

print(json.dumps(manifest, indent=2)[:16000])
print("Memo path:", mri_path)
display(pd.DataFrame(memos[:10]))
